### Самый честный способ создания диффузионной задачи

In [ ]:
from yggdrasill import Hypergraph
from yggdrasill.engine.edge import Edge
from yggdrasill.integrations.diffusers import contracts as C
from yggdrasill.integrations.diffusers.model_store import ModelStore

store = ModelStore.default()
repo = "runwayml/stable-diffusion-v1-5"
components = store.load_components_by_keys(
    "sd15", ["tokenizer", "text_encoder", "unet", "vae", "scheduler"],
    repo, variant="fp16", torch_dtype="float16",
)

graph = Hypergraph(name="MySD15Graph")
cfg = {"height": 512, "width": 512, "num_inference_steps": 50, "device": "cuda", "guidance_scale": 7.5}

graph.add_node("tokenizer", type="sd15/tokenizer", config={"tokenizer": components["tokenizer"]})
graph.add_node("prompt_enc", type="sd15/prompt_encoder", config={
    "text_encoder": components["text_encoder"],
})
graph.add_node("sched_setup", type="sd15/scheduler_setup", config={
    "scheduler": components["scheduler"],
    "num_inference_steps": cfg["num_inference_steps"],
    "device": cfg["device"],
})
graph.add_node("latent_init", type="sd15/latent_init", config={
        "height": cfg["height"], "width": cfg["width"],
        "device": cfg["device"], "dtype": "float16",
})
graph.add_node("unet", type="sd15/unet", config={
    "unet": components["unet"],
    "guidance_scale": cfg["guidance_scale"],
})
graph.add_node("sched_step", type="sd15/scheduler_step", config={"scheduler": components["scheduler"]})
graph.add_node("vae_decode", type="sd15/vae_decode", config={
    "vae": components["vae"],
    "output_type": "pil",
})

# Edges
graph.add_edge(Edge("tokenizer", C.PORT_INPUT_IDS, "prompt_enc", C.PORT_INPUT_IDS))
graph.add_edge(Edge("tokenizer", C.PORT_NEGATIVE_INPUT_IDS, "prompt_enc", C.PORT_NEGATIVE_INPUT_IDS))
graph.add_edge(Edge("sched_setup", C.PORT_SCHEDULER_STATE, "latent_init", C.PORT_SCHEDULER_STATE))
graph.add_edge(Edge("sched_setup", C.PORT_SCHEDULER_STATE, "unet", C.PORT_SCHEDULER_STATE))
graph.add_edge(Edge("prompt_enc", C.PORT_PROMPT_EMBEDS, "unet", C.PORT_PROMPT_EMBEDS))
graph.add_edge(Edge("prompt_enc", C.PORT_NEGATIVE_PROMPT_EMBEDS, "unet", C.PORT_NEGATIVE_PROMPT_EMBEDS))
graph.add_edge(Edge("latent_init", C.PORT_LATENTS, "unet", C.PORT_LATENTS))
graph.add_edge(Edge("latent_init", C.PORT_LATENTS, "sched_step", C.PORT_LATENTS))
graph.add_edge(Edge("latent_init", C.PORT_TIMESTEP, "unet", C.PORT_TIMESTEP))
graph.add_edge(Edge("latent_init", C.PORT_TIMESTEP, "sched_step", C.PORT_TIMESTEP))
graph.add_edge(Edge("unet", C.PORT_NOISE_PRED, "sched_step", C.PORT_NOISE_PRED))
graph.add_edge(Edge("sched_step", "next_latent", "unet", C.PORT_LATENTS))
graph.add_edge(Edge("sched_step", "next_latent", "sched_step", C.PORT_LATENTS))
graph.add_edge(Edge("sched_step", "next_timestep", "unet", C.PORT_TIMESTEP))
graph.add_edge(Edge("sched_step", "next_timestep", "sched_step", C.PORT_TIMESTEP))
graph.add_edge(Edge("sched_step", "next_latent", "vae_decode", C.PORT_LATENTS))

graph.expose_input("tokenizer", C.PORT_PROMPT, C.PORT_PROMPT)
graph.expose_input("tokenizer", C.PORT_NEGATIVE_PROMPT, C.PORT_NEGATIVE_PROMPT)
graph.expose_output("vae_decode", C.PORT_DECODED_IMAGE, C.PORT_OUTPUT_IMAGE)
graph.metadata["num_loop_steps"] = cfg["num_inference_steps"]

graph.to("cuda")

In [ ]:
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Создание через специализированный DiffusionGraphBuilder

In [ ]:
from yggdrasill import DiffusionGraphBuilder

repo = "runwayml/stable-diffusion-v1-5"

builder = DiffusionGraphBuilder(name="MySD15Graph")
builder.add_component("Backbone", "sd15.unet", pretrained=repo)
builder.add_component("Scheduler", "sd15.scheduler", pretrained=repo)
builder.add_component("Tokenizer", "sd15.tokenizer", pretrained=repo)
builder.add_component("TextEncoder", "sd15.text_encoder", pretrained=repo)
builder.add_component("Autoencoder", "sd15.vae", pretrained=repo)
builder.to("cuda")

In [ ]:
output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Использование в 2 строчки кода

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_text2image", device="cuda")

In [ ]:
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
# import torch
# from diffusers import StableDiffusionPipeline

# pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)

# pipe.to("cuda")

# output = pipe(
#     prompt="A photo of an astronaut riding a horse on mars",
#     negative_prompt="blurry",
#     num_inference_steps=50,
#     guidance_scale=7.5,
#     num_images_per_prompt=1,
#     generator=torch.Generator(device="cuda").manual_seed(42),
#     width=512,
#     height=512,
# )

# output.images[0]

### Добавление новых компонент в граф

In [ ]:
from yggdrasill import Hypergraph, DiffusionGraphBuilder

graph = Hypergraph.from_template("sd15_text2image", repo_id="Lykon/DreamShaper", device="cuda")

builder = DiffusionGraphBuilder(graph)
builder.add_component("CannyControlNet", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")
builder.add_component("DepthControlNet", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-depth")
builder.add_component("IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter")
builder.to("cuda")

graph = builder.graph

In [ ]:
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    controlnet_image={
        "CannyControlNet": "https://media.licdn.com/dms/image/v2/C5112AQExoUiRNUfNyg/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1582133550471?e=2147483647&v=beta&t=SAvSWpaMfcrgAB_tWPHZkrvFEGRXKSKo-fuiORZU16I",
        "DepthControlNet": "https://www.cyanilux.com/tutorials/depth/Preview.png",
    },
    controlnet_conditioning_scale={
        "CannyControlNet": 0.9,
        "DepthControlNet": 0.9,
    },
    ip_adapter_image={
        "IPAdapter": "https://res.cloudinary.com/jerrick/image/upload/d_642250b563292b35f27461a7.png,f_jpg,fl_progressive,q_auto,w_1024/63dfe49d72df38001c5175bc.jpg",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.75,
    },
    num_inference_steps=30,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Полная сборка с доп компонентами

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder(name="MySD15Graph")
builder.add_component("Backbone", "sd15.unet", pretrained="Lykon/DreamShaper")
builder.add_component("Scheduler", "sd15.scheduler", pretrained="Lykon/DreamShaper")
builder.add_component("Tokenizer", "sd15.tokenizer", pretrained="Lykon/DreamShaper")
builder.add_component("TextEncoder", "sd15.text_encoder", pretrained="Lykon/DreamShaper")
builder.add_component("Autoencoder", "sd15.vae", pretrained="Lykon/DreamShaper")
builder.add_component("CannyControlNet", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")
builder.add_component("DepthControlNet", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-depth")
builder.add_component(
    "IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin"
)
builder.to("cuda")

In [ ]:
output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    image="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQCg6QUI0p_a6cAgyGzKZM8q4c4kJwo1eEuCw&s",
    strength=0.9,
    controlnet_image={
        "CannyControlNet": "https://media.licdn.com/dms/image/v2/C5112AQExoUiRNUfNyg/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1582133550471?e=2147483647&v=beta&t=SAvSWpaMfcrgAB_tWPHZkrvFEGRXKSKo-fuiORZU16I",
        "DepthControlNet": "https://www.cyanilux.com/tutorials/depth/Preview.png",
    },
    controlnet_conditioning_scale={
        "CannyControlNet": 0.9,
        "DepthControlNet": 0.9,
    },
    ip_adapter_image={
        "IPAdapter": "https://res.cloudinary.com/jerrick/image/upload/d_642250b563292b35f27461a7.png,f_jpg,fl_progressive,q_auto,w_1024/63dfe49d72df38001c5175bc.jpg",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.75,
    },
    num_inference_steps=30,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Замена компонентов в графе

In [ ]:
from yggdrasill import Hypergraph, DiffusionGraphBuilder

graph = Hypergraph.from_template("sd15_text2image", device="cuda")

builder = DiffusionGraphBuilder(graph)
builder.replace_component("Backbone", "sd15.unet", pretrained="Lykon/DreamShaper")

In [ ]:
output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
builder.replace_component("Scheduler", "sd15.scheduler", scheduler_type="euler")

output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
builder.add_component("ControlNet", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")

output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    controlnet_image={
        "ControlNet": "https://psv4.userapi.com/s/v1/d2/m_5kQALU9DlVvJ0xyyPHbhaaMFjJUlp_ovyz7p6vgPjeXd7yoTQ_VhG9a8YhQSXRw4KpwQD4sz_xoW79yuCvadoM8bbBMVD_8DWKhuLNjziAZbHOP--nd4ojZ9HokQ9527NWeCC8BIqI/1774007112940.jpg",
    },
    controlnet_conditioning_scale={
        "ControlNet": 0.5,
    },
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
builder.add_component("IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter")

output = builder.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    controlnet_image={
        "ControlNet": "https://psv4.userapi.com/s/v1/d2/m_5kQALU9DlVvJ0xyyPHbhaaMFjJUlp_ovyz7p6vgPjeXd7yoTQ_VhG9a8YhQSXRw4KpwQD4sz_xoW79yuCvadoM8bbBMVD_8DWKhuLNjziAZbHOP--nd4ojZ9HokQ9527NWeCC8BIqI/1774007112940.jpg",
    },
    controlnet_conditioning_scale={
        "ControlNet": 0.5,
    },
    ip_adapter_image={
        "IPAdapter": "https://images.stroistyle.com/posts/9467675-veranda-iz-kirpicha-2.jpg",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.75,
    },
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
builder.add_component("ControlNetScribble", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-scribble")

output = builder.run(
    prompt="A woman with a red dress",
    negative_prompt="blurry",
    controlnet_image={
        "ControlNetScribble": "https://huggingface.co/takuma104/controlnet_dev/resolve/main/gen_compare/control_images/converted/control_vermeer_scribble.png",
    },
    controlnet_conditioning_scale={
        "ControlNetScribble": 0.9,
    },
    # ip_adapter_image={
    #     "IPAdapter": "https://images.stroistyle.com/posts/9467675-veranda-iz-kirpicha-2.jpg",
    # },
    # ip_adapter_conditioning_scale={
    #     "IPAdapter": 0.75,
    # },
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Проверка работы img2img

In [ ]:
from yggdrasill import Hypergraph, DiffusionGraphBuilder

graph = Hypergraph.from_template("sd15_img2img", device="cuda")
builder = DiffusionGraphBuilder(graph)
builder.replace_component("Backbone", "sd15.unet", pretrained="Lykon/DreamShaper")
graph = builder.graph

output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    image="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRtxLSt4-D6-Tps1EIHblCehsP0yB5evpwylw&s",
    strength=0.9,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Проверка работы inpaint

In [ ]:
from yggdrasill import Hypergraph, DiffusionGraphBuilder

graph = Hypergraph.from_template("sd15_inpaint", device="cuda")

builder = DiffusionGraphBuilder(graph)
builder.replace_component("Backbone", "sd15.unet", pretrained="Lykon/DreamShaper")
graph = builder.graph

output = graph.run(
    prompt="concept art digital painting of an elven castle, inspired by lord of the rings, highly detailed, 8k",
    negative_prompt="blurry",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png",
    strength=1.0,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Проверка работы универсального графа

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder(name="MySD15Graph")
builder.add_component("Backbone", "sd15.unet", pretrained="Lykon/DreamShaper")
builder.add_component("Scheduler", "sd15.scheduler", pretrained="Lykon/DreamShaper")
builder.add_component("Tokenizer", "sd15.tokenizer", pretrained="Lykon/DreamShaper")
builder.add_component("TextEncoder", "sd15.text_encoder", pretrained="Lykon/DreamShaper")
builder.add_component("Autoencoder", "sd15.vae", pretrained="Lykon/DreamShaper")
builder.to("cuda")

In [ ]:
output = builder.run(
    prompt="concept art digital painting of an elven castle, inspired by lord of the rings, highly detailed, 8k",
    negative_prompt="blurry",
    # image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    # # mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png",
    # strength=0.9,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Проверка что отдельные пайплайны выполняют только свою задачу 

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_text2image", device="cuda")

output = graph.run(
    prompt="concept art digital painting of an elven castle, inspired by lord of the rings, highly detailed, 8k",
    negative_prompt="blurry",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png",
    strength=0.2,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_img2img", device="cuda")

output = graph.run(
    prompt="concept art digital painting of an elven castle, inspired by lord of the rings, highly detailed, 8k",
    negative_prompt="blurry",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png",
    strength=0.7,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_inpaint", device="cuda")

output = graph.run(
    prompt="concept art digital painting of an elven castle, inspired by lord of the rings, highly detailed, 8k",
    negative_prompt="blurry",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png",
    strength=0.9,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

# Полная поддержка IP-Adapter

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component(
    "IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin"
)

output = builder.run(
    prompt="a polar bear sitting in a chair drinking a milkshake",
    negative_prompt="deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality",
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_diner.png"
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.9,
    },
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_img2img", device="cuda")
builder.add_component(
    "IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin"
)

output = builder.run(
    prompt="best quality, high quality",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_bear_1.png",
    strength=0.5,
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_gummy.png",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.3,
    },
)

output.images[0]

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_inpaint", device="cuda")
builder.add_component(
    "IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin"
)

output = builder.run(
    prompt="a cute gummy bear waving",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_bear_1.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_mask.png",
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_gummy.png",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.9,
    },
)

output.images[0]

### IP-Adapter: предвычисленные эмбеддинги

In [ ]:
import torch
from diffusers.utils import load_image
from yggdrasill import DiffusionGraphBuilder

from yggdrasill.integrations.diffusers import prepare_ip_adapter_image_embeds

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component(
    "IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin"
)

ref = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_diner.png"
)

ref_embeds = prepare_ip_adapter_image_embeds(
    builder.graph,  
    ref,
    device="cuda",
    num_images_per_prompt=1,
    do_classifier_free_guidance=True,
)
torch.save(ref_embeds, "image_embeds.ipadpt")

loaded = torch.load("image_embeds.ipadpt", map_location="cuda")
# один слот IP-Adapter → один тензор в списке
emb_arg = loaded[0] if len(loaded) == 1 else loaded

output = builder.run(
    prompt="a polar bear sitting in a chair drinking a milkshake",
    negative_prompt="deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality",
    ip_adapter_image_embeds={"IPAdapter": emb_arg},
    ip_adapter_conditioning_scale={"IPAdapter": 0.8},
    guidance_scale=7.5,
    num_inference_steps=50,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)
output.images[0]

### Маски с IP-Adapter

In [ ]:
from yggdrasill import DiffusionGraphBuilder

mask_builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
mask_builder.add_component(
    "IPAdapter",
    "sd15.ipadapter",
    pretrained="h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter-plus-face_sd15.safetensors",
)

out = mask_builder.run(
    prompt="2 girls",
    negative_prompt="monochrome, lowres, bad anatomy, worst quality, low quality",
    ip_adapter_image={"IPAdapter": [
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_mask_girl2.png",
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_mask_girl1.png",
    ]},
    ip_adapter_mask_images=[
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_mask_mask2.png",
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_mask_mask1.png",
    ],
    ip_adapter_conditioning_scale={"IPAdapter": [0.7, 0.7]},
    seed=42,
)

out.images[0]

### Face models

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component(
    "IPAdapter", 
    "sd15.ipadapter", 
    pretrained="h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter-plus-face_sd15.safetensors",    
)

builder.run(
    prompt="A photo of Einstein as a chef, wearing an apron, cooking in a French restaurant",
    negative_prompt="lowres, bad anatomy, worst quality, low quality",
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_einstein_base.png",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.5,
    },
).images[0]

### Multiple IP-Adapters

In [ ]:
from diffusers.utils import load_image
from yggdrasill import DiffusionGraphBuilder

b = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")

b.add_component(
    "style_ip",
    "sd15.ipadapter",
    pretrained="h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter-plus_sd15.safetensors",
)
b.add_component(
    "face_ip",
    "sd15.ipadapter",
    pretrained="h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter-plus-face_sd15.safetensors",
)

style_folder = "https://huggingface.co/datasets/YiYiXu/testing-images/resolve/main/style_ziggy"
style_images = [load_image(f"{style_folder}/img{i}.png") for i in range(10)]
face_image = load_image("https://huggingface.co/datasets/YiYiXu/testing-images/resolve/main/women_input.png")

out = b.run(
    prompt="wonderwoman",
    ip_adapter_image={
        "style_ip": style_images,
        "face_ip": face_image,
    },
    ip_adapter_conditioning_scale={
        "style_ip": 0.7,
        "face_ip": 0.3,
    },
    negative_prompt="monochrome, lowres, bad anatomy, worst quality, low quality",
    seed=42,
)

out.images[0]

### Structural control

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("ControlNetDepth", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-depth")
builder.add_component("IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

out = builder.run(
    prompt="best quality, high quality",
    negative_prompt="monochrome, lowres, bad anatomy, worst quality, low quality",
    controlnet_image={
        "ControlNetDepth": "https://huggingface.co/datasets/YiYiXu/testing-images/resolve/main/depth.png",
    },
    controlnet_conditioning_scale={
        "ControlNetDepth": 0.9,
    },
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/YiYiXu/testing-images/resolve/main/statue.png",
    },
    ip_adapter_conditioning_scale={
        "IPAdapter": 0.9,
    },
)

out.images[0]

### Style and layout control

In [ ]:
from yggdrasill import DiffusionGraphBuilder

instant_style = {
    "down": {"block_2": [0.0, 1.0]},
    "up": {"block_0": [0.0, 1.0, 0.0]},
}

mask_builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
mask_builder.add_component("IPAdapter", "sd15.ipadapter", pretrained="h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

out = mask_builder.run(
    prompt="a cat, masterpiece, best quality, high quality",
    ip_adapter_image={
        "IPAdapter": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg",
    },
    # ip_adapter_conditioning_scale=instant_style,
    ip_adapter_conditioning_scale={"IPAdapter": 0.7},
    guidance_scale=5.0,
)

out.images[0]

# Полная поддержка ControlNet

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("ControlNetCanny", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")

builder.run(
    prompt="""
    A photorealistic overhead image of a cat reclining sideways in a flamingo pool floatie holding a margarita. 
    The cat is floating leisurely in the pool and completely relaxed and happy.
    """,
    controlnet_image={
        "ControlNetCanny": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/canny-cat.png",
    },
    controlnet_conditioning_scale={
        "ControlNetCanny": 0.5,
    },
    num_inference_steps=50,
    guidance_scale=3.5,
).images[0]

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_img2img", device="cuda")
builder.add_component("ControlNetDepth", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-depth")

builder.run(
    prompt = """
    A photorealistic overhead image of a cat reclining sideways in a flamingo pool floatie holding a margarita. 
    The cat is floating leisurely in the pool and completely relaxed and happy.
    """,
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/non-enhanced-prompt.png",
    strength=0.99,
    controlnet_image={
        "ControlNetDepth": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/sdxl_depth_image.png",
    },
    controlnet_conditioning_scale={
        "ControlNetDepth": 0.5,
    },
).images[0]


In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_inpaint", device="cuda")
builder.add_component("ControlNetCanny", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")

builder.run(
    prompt="a cute and fluffy bunny rabbit",
    image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/non-enhanced-prompt.png",
    mask_image="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat_mask.png",
    strength=0.99,
    controlnet_image={
        "ControlNetCanny": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/canny-cat.png",
    },
    controlnet_conditioning_scale={
        "ControlNetCanny": 0.5,
    },
).images[0]

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("ControlNetCanny", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")
builder.add_component("ControlNetDepth", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-depth")

builder.run(
    prompt = """
    a relaxed rabbit sitting on a striped towel next to a pool with a tropical drink nearby, 
    bright sunny day, vacation scene, 35mm photograph, film, professional, 4k, highly detailed
    """,
    negative_prompt = "lowres, bad anatomy, worst quality, low quality, deformed, ugly",
    controlnet_image={
        "ControlNetCanny": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/canny-cat.png",
        "ControlNetDepth": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/multicontrolnet_depth.png",
    },
    controlnet_conditioning_scale={
        "ControlNetCanny": 0.5,
        "ControlNetDepth": 0.5,
    },
).images[0]


In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("ControlNetCanny", "sd15.controlnet", pretrained="lllyasviel/sd-controlnet-canny")

builder.run(
    prompt="",
    controlnet_image={
        "ControlNetCanny": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/canny-cat.png",
    },
    guess_mode=True,
    seed=42,
).images[0]

# Поддержка T2I-Adapter

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("T2IAdapter", "sd15.t2iadapter", pretrained="TencentARC/t2iadapter_canny_sd15v2")

builder.run(
    prompt = """
    A photorealistic overhead image of a cat reclining sideways in a flamingo pool floatie holding a margarita. 
    The cat is floating leisurely in the pool and completely relaxed and happy.
    """,
    t2i_adapter_image={
        "T2IAdapter": "https://camo.githubusercontent.com/3723391be9a2d01204d13d5db6b8d4eb90ee74407ab939ee0c41744e44321a07/68747470733a2f2f68756767696e67666163652e636f2f64617461736574732f68756767696e67666163652f646f63756d656e746174696f6e2d696d616765732f7265736f6c76652f6d61696e2f6469666675736572732f63616e6e792d6361742e706e67",
    },
    t2i_adapter_conditioning_scale={
        "T2IAdapter": 0.9,
    },
    seed=42,
).images[0]

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.add_component("T2IAdapterCanny", "sd15.t2iadapter", pretrained="TencentARC/t2iadapter_canny_sd15v2")
builder.add_component("T2IAdapterDepth", "sd15.t2iadapter", pretrained="TencentARC/t2iadapter_depth_sd15v2")

builder.run(
    prompt = ["""
    a relaxed rabbit sitting on a striped towel next to a pool with a tropical drink nearby, 
    bright sunny day, vacation scene, 35mm photograph, film, professional, 4k, highly detailed
    """],
    t2i_adapter_image={
        "T2IAdapterCanny": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/canny-cat.png",
        "T2IAdapterDepth": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/sdxl_depth_image.png",
    },
    t2i_adapter_conditioning_scale={
        "T2IAdapterCanny": 0.7,
        "T2IAdapterDepth": 0.7,
    },
).images[0]

# Поддержка LoRA

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_text2img", device="cuda")
builder.replace_component("Backbone", "sd15.unet", pretrained="Lykon/NeverEnding-Dream")
builder.add_component(
    "LoRA1", "sd15.lora", pretrained="GrafikXxxxxxxYyyyyyyyyyy/loras", subfolder="sd15", weight_name="makima.safetensors"
)
builder.add_component(
    "LoRA2", "sd15.lora", pretrained="GrafikXxxxxxxYyyyyyyyyyy/loras", subfolder="sd15", weight_name="ahegao.safetensors"
)

builder.run(
    prompt = "masterpiece, (photorealistic:1.4), best quality, beautiful lighting, (ulzzang-6500:0.5), makima \(chainsaw man\), (red hair)+(long braided hair)+(bangs), yellow eyes, RAW photo, 8k uhd," + "ahegao, rolling_eyes, cumshot",
    negative_prompt = "lowres, bad anatomy, worst quality, low quality, bad face, deformed, ugly, bad eyes",
    lora_conditioning_scale={
        "LoRA1": 0.7,
        "LoRA2": 0.7,
    },
    seed=42,
).images[0]